# Activation Geometry

Clone the repo into Google Colab, validate that the locally generated Gemini-backed prompt set and split manifests are present, then measure last-token residual activations on the discovery split.

**Runtime:** T4 GPU

In [1]:
import os
import subprocess
import sys
from getpass import getpass
from pathlib import Path

try:
    from google.colab import userdata as colab_userdata
except ImportError:
    colab_userdata = None


def read_secret(name: str) -> str:
    if colab_userdata is not None:
        try:
            return colab_userdata.get(name).strip()
        except Exception:
            pass
    return os.environ.get(name, "").strip()


REPO_URL = "https://github.com/aaliyan1230/false-refusal-suppression.git"
HF_TOKEN = read_secret("HF_TOKEN") or getpass("Enter your HF token (or press Enter to skip): ")
REPO_DIR = Path("/content/false-refusal-suppression")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = HF_TOKEN

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-e", f"{REPO_DIR}[train,dev]"], check=True)

os.chdir(REPO_DIR)
src_path = REPO_DIR / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(REPO_DIR)
print("HF token loaded:", bool(HF_TOKEN))

/content/false-refusal-suppression
HF token loaded: True


In [ ]:
from pathlib import Path

MODEL_ID = "Qwen/Qwen3-4B"
PILOT_PROMPTS = Path("data/processed/prompts/pilot_prompts_gemini.jsonl")
SPLIT_DIR = Path("data/processed/splits/pilot_gemini")
ACTIVATION_ARTIFACT = Path("artifacts/activations/qwen3_gemini_pilot_discovery.json")
DIRECTION_ARTIFACT = Path("artifacts/directions/qwen3_gemini_pilot_borderline_vs_easy.json")

required_paths = [PILOT_PROMPTS, SPLIT_DIR / "discovery.jsonl"]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing prebuilt local data artifacts. Generate and commit them locally before using Kaggle: " + ", ".join(missing)
    )

print(MODEL_ID)
print(PILOT_PROMPTS)
print(ACTIVATION_ARTIFACT)
print(DIRECTION_ARTIFACT)

Qwen/Qwen3-4B
data/processed/prompts/pilot_prompts_gemini.jsonl
artifacts/activations/qwen3_gemini_pilot_discovery.json
artifacts/directions/qwen3_gemini_pilot_borderline_vs_easy.json


## Baseline Refusal Audit

Before computing any directions, check whether the **unedited** model actually refuses the `benign_borderline` prompts. If it doesn't, the "false refusal" label is ungrounded and the direction will capture topic noise instead of refusal behavior.

In [4]:
import gc
import torch
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from frs.data.loaders import load_prompt_examples
from frs.evaluation.refusal import is_refusal
from frs.models.generation import TextGenerationConfig, generate_text

gc.collect()
torch.cuda.empty_cache()

# Load model with explicit GPU placement
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},
)
model.eval()

discovery_examples = load_prompt_examples(str(SPLIT_DIR / "discovery.jsonl"))
gen_config = TextGenerationConfig(max_new_tokens=256, temperature=0.0)

baseline_records = []
for i, ex in enumerate(discovery_examples):
    response = generate_text(model, tokenizer, ex.prompt, gen_config)
    refused = is_refusal(response)
    baseline_records.append({
        "id": ex.id,
        "group": ex.group,
        "topic": ex.topic,
        "prompt": ex.prompt[:120],
        "refused": refused,
        "response_start": response[:200],
    })
    if (i + 1) % 10 == 0:
        print(f"  [{i+1}/{len(discovery_examples)}] generated")

# Free GPU for later cells
del model, tokenizer
gc.collect()
torch.cuda.empty_cache()

baseline_df = pd.DataFrame(baseline_records)

# Per-group refusal rates
refusal_rates = (
    baseline_df.groupby("group")["refused"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "refusals", "count": "total"})
)
refusal_rates["refusal_rate"] = refusal_rates["refusals"] / refusal_rates["total"]
display(refusal_rates)

# Show borderline prompts and whether they were actually refused
borderline = baseline_df[baseline_df["group"] == "benign_borderline"]
display(borderline[["prompt", "refused", "response_start"]])

actual_false_refusals = borderline["refused"].sum()
print(f"\nActual false refusals on borderline prompts: {actual_false_refusals}/{len(borderline)}")
if actual_false_refusals == 0:
    print("WARNING: Model does NOT refuse any borderline prompts.")
    print("The current prompt set needs harder borderline examples that actually trigger refusals.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  [10/64] generated
  [20/64] generated


KeyboardInterrupt: 

In [3]:
subprocess.run(
    [
        sys.executable,
        "scripts/measure_activations.py",
        "--model-id",
        MODEL_ID,
        "--split-path",
        str(SPLIT_DIR / "discovery.jsonl"),
        "--output",
        str(ACTIVATION_ARTIFACT),
        "--capture-default-modules",
        "--max-module-captures",
        "12",
    ],
    check=True,
 )

subprocess.run(
    [
        sys.executable,
        "scripts/compute_directions.py",
        "--activations",
        str(ACTIVATION_ARTIFACT),
        "--source-group-a",
        "benign_borderline",
        "--source-group-b",
        "benign_easy",
        "--output",
        str(DIRECTION_ARTIFACT),
    ],
    check=True,
 )

CompletedProcess(args=['/usr/bin/python3', 'scripts/compute_directions.py', '--activations', 'artifacts/activations/qwen3_gemini_pilot_discovery.json', '--source-group-a', 'benign_borderline', '--source-group-b', 'benign_easy', '--output', 'artifacts/directions/qwen3_gemini_pilot_borderline_vs_easy.json'], returncode=0)

In [3]:
import json
import matplotlib.pyplot as plt
import pandas as pd

with open(ACTIVATION_ARTIFACT, "r", encoding="utf-8") as handle:
    activation_artifact = json.load(handle)

with open(DIRECTION_ARTIFACT, "r", encoding="utf-8") as handle:
    direction_artifact = json.load(handle)

ranked_layers = pd.DataFrame(direction_artifact["ranked_layers"])
display(pd.DataFrame([activation_artifact["summary"]]))
display(ranked_layers.head(12))

if not ranked_layers.empty:
    ax = ranked_layers.head(12).plot.bar(
        x="name",
        y="score",
        figsize=(10, 4),
        legend=False,
        title="Top separability scores by layer",
    )
    ax.set_ylabel("separability score")
    plt.xticks(rotation=45, ha="right")
    plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'artifacts/activations/qwen3_gemini_pilot_discovery.json'